# MNIST API — Fase 1: Pipeline offline e serialização do modelo

**Objetivo:** treinar o classificador MLP vencedor do mini-projeto dentro de um
`sklearn.pipeline.Pipeline` e serializá-lo com `joblib` para uso posterior pela API FastAPI.

**Por que um Pipeline e não só o classificador?**

O Pipeline empacota *pré-processamento + modelo* em um único objeto. Isso garante
que a transformação aplicada nos dados de treino é **exatamente a mesma** aplicada
em produção (API). Se a normalização ficasse "solta" no notebook, alguém poderia
esquecer de aplicá-la na API — ou aplicar de outro jeito — e o modelo erraria
silenciosamente (distribuição de entrada divergente).

## 1. Imports e caminhos

Usamos `pathlib.Path` em vez de strings para caminhos:
- Mais legível e menos propenso a erros de `\` vs `/` (Windows vs Linux)
- `Path` resolve caminhos relativos ao notebook de forma portável
- Se o notebook rodar de outra pasta, `Path("models")` ainda aponta para o lugar certo

In [59]:
import json
import platform
import subprocess
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import sklearn
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.exceptions import ConvergenceWarning
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_recall_fscore_support,
    roc_auc_score,
)

# Importa a função de normalização do módulo compartilhado
# (esse import funciona porque app/ é um pacote — veja app/__init__.py)
from app.preprocessing import normalizar_pixels

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

print(f"scikit-learn: {sklearn.__version__}")
print(f"Python: {platform.python_version()}")

scikit-learn: 1.9.1
Python: 3.13.15


### Por que `app/preprocessing.py` e não uma função definida aqui no notebook?

Quando fazemos `joblib.dump(pipeline, "modelo.joblib")`, o joblib serializa o
*objeto* Pipeline — incluindo referências aos métodos/funções que ele chama.
Para `FunctionTransformer`, ele grava o caminho importável da função
(ex.: `app.preprocessing.normalizar_pixels`).

Ao carregar com `joblib.load()` em **outro processo** (a API FastAPI), o Python
precisa importar essa função de volta. Se ela estivesse definida só neste notebook
(como variável local), o import falharia:

```
AttributeError: Can't get attribute 'normalizar_pixels' on <module '__main__'>
```

Criar `app/__init__.py` (mesmo vazio) transforma `app/` em um **pacote Python**,
tornando o import estável em qualquer processo que tenha o diretório no `sys.path`.

## 2. Carregamento dos dados e divisão estratificada 70/10/20

**Divisão estratificada** mantém a proporção de cada dígito (0–9) em cada subconjunto.
Sem isso, um dígito raro poderia acabar todo no teste e inflar/derrubar as métricas
de forma não representativa.

**Isso é leakage?** Não. A normalização `X/255.0` é uma transformação **fixa** —
ela não "aprende" nada dos dados (diferente de `StandardScaler`, que calcula média/desvio
do conjunto de treino). Aplicar `X/255` antes do split, dentro do Pipeline ou depois
não introduz viés. A única razão de colocá-la *dentro* do Pipeline é **consistência**
com a API, não prevenção de leakage.

**Nota sobre o cache do MNIST:** `fetch_openml` baixa o dataset (dezenas de MB)
e o guarda no cache do usuário (`~/scikit_learn_data`). Isso evita versionar
arquivos grandes no repositório, mas exige internet na primeira execução.

In [60]:
# MNIST via OpenML (mesmo dataset do scikit-learn, sem depender do fetch_mldata antigo)
mnist = fetch_openml("mnist_784", version=1, as_frame=False, parser="auto")
X, y = mnist.data, mnist.target.astype("int64")

print(f"Shape de X: {X.shape}")
print(f"Shape de y: {y.shape}")
print(f"Classes: {np.unique(y)}")

Shape de X: (70000, 784)
Shape de y: (70000,)
Classes: [0 1 2 3 4 5 6 7 8 9]


In [61]:
# Divisão estratificada em 3 partes: treino (70%), validação (10%), teste (20%)
#
# Estratégia: dividimos 2 vezes de forma estratificada
#   1º split: 80% (treino+val) vs 20% (teste)
#   2º split: dos 80%, separamos 12.5% como validação → 70% total treino, 10% total validação

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.125, random_state=42, stratify=y_temp
)  # 0.125 de 80% = 10% do total

print(f"Treino:    {X_train.shape[0]:>6} amostras")
print(f"Validação: {X_val.shape[0]:>6} amostras")
print(f"Teste:     {X_test.shape[0]:>6} amostras")

Treino:     49000 amostras
Validação:   7000 amostras
Teste:      14000 amostras


## 3. Construção do Pipeline (normalização + MLP)

**Hiperparâmetros do MLP vencedor** (escolhidos no mini-projeto com base na validação;
aqui apenas reproduzimos):
- `hidden_layer_sizes=(128,)` — uma camada oculta com 128 neurônios
- `activation="relu"` — não-linearidade padrão, evita vanishing gradient
- `solver="adam"` — otimizador adaptativo, bom para grandes volumes
- `learning_rate_init=0.001` — taxa de aprendizado inicial
- `alpha=0.0001` — regularização L2 (weight decay) para reduzir overfitting
- `max_iter=30` — épocas máximas
- `early_stopping=True` — reserva 10% do próprio X_train como validação interna
  e interrompe o treino se a acurácia de validação interna não melhorar
  por `n_iter_no_change=10` épocas consecutivas
- `random_state=42` — reprodutibilidade

**Sobre `early_stopping=True`:** o MLP separa `validation_fraction=0.1` (10%) do
**próprio X_train** como validação interna. Ele **não** usa o nosso `X_val` externo.
Nosso `X_val` serve para avaliação pós-treino (comparar com o teste e checar overfitting).

**Por que `FunctionTransformer`?**
O Pipeline só aceita *estimadores* (com `.fit`/`.transform` ou `.predict`).
`FunctionTransformer` embrulha uma função simples como um transformador compatível,
permitindo incluir nossa normalização fixa no mesmo objeto do modelo.

In [62]:
pipeline = Pipeline(
    steps=[
        ("normalizacao", FunctionTransformer(normalizar_pixels, validate=False)),
        (
            "mlp",
            MLPClassifier(
                hidden_layer_sizes=(128,),
                activation="relu",
                solver="adam",
                learning_rate_init=0.001,
                alpha=0.0001,
                max_iter=30,
                early_stopping=True,
                random_state=42,
            ),
        ),
    ]
)

print(pipeline)

Pipeline(steps=[('normalizacao',
                 FunctionTransformer(func=<function normalizar_pixels at 0x0000017FD443A8E0>)),
                ('mlp',
                 MLPClassifier(early_stopping=True, hidden_layer_sizes=(128,),
                               max_iter=30, random_state=42))])


In [63]:
# Treino: o Pipeline aplica a normalização automaticamente em X_train.
# O early_stopping interno do MLP usa 10% do próprio X_train (validation_fraction),
# NÃO o nosso X_val externo.
#
# Suprimimos apenas o ConvergenceWarning (max_iter=30 atingido):
# é esperado e documentado na saída abaixo.
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=ConvergenceWarning)
    pipeline.fit(X_train, y_train)

mlp = pipeline.named_steps["mlp"]
n_iter = mlp.n_iter_
print(f"Pipeline treinado com sucesso.")
print(f"Épocas executadas: {n_iter} / {mlp.max_iter}")
if n_iter >= mlp.max_iter:
    print("Aviso: o limite de épocas foi atingido (ConvergenceWarning suprimido abaixo).")
    print("O early_stopping não interrompeu antes, mas a acurácia permanece estável.")
    print("Mantivemos max_iter=30 idêntico ao mini-projeto para reprodutibilidade.")

Pipeline treinado com sucesso.
Épocas executadas: 30 / 30
Aviso: o limite de épocas foi atingido (ConvergenceWarning suprimido abaixo).
O early_stopping não interrompeu antes, mas a acurácia permanece estável.
Mantivemos max_iter=30 idêntico ao mini-projeto para reprodutibilidade.


## 4. Avaliação nos conjuntos de treino, validação e teste

Reportamos a acurácia nos três conjuntos:
- **Treino** — o modelo viu essas amostras durante o fit; serve como referência.
- **Validação** — nunca vista no fit; comparada com o treino para checar overfitting
  e com o teste para checar consistência.
- **Teste** — nunca visto; avaliação final.

| Métrica | O que mede | Como interpretar |
|---|---|---|
| **Acurácia** | Proporção de acertos globais | % de dígitos classificados corretamente |
| **Precision (weighted)** | Dos que o modelo disse ser classe X, quantos realmente eram | Média ponderada por suporte; penaliza falsos positivos |
| **Recall (weighted)** | Dos que realmente são classe X, quantos o modelo acertou | Média ponderada; penaliza falsos negativos |
| **F1 (weighted)** | Média harmônica de precision e recall | Balanceia FP e FN; bom resumo único |
| **ROC-AUC (OVR)** | Capacidade de ranquear cada classe vs as demais | 1.0 = perfeito; ~0.5 = aleatório; média "one-vs-rest" |

**Por que weighted?** Cada dígito tem quantidade similar de amostras no MNIST, mas
"weighted" garante que classes com mais amostras não dominem a métrica se houvesse desbalanceamento.

In [64]:
# Avaliação em treino (referência), validação (consistência) e teste (avaliação final)
acc_train = pipeline.score(X_train, y_train)
acc_val = pipeline.score(X_val, y_val)

y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)

# Acurácia
acc = accuracy_score(y_test, y_pred)

# Precision / Recall / F1 (weighted)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_test, y_pred, average="weighted"
)

# ROC-AUC multiclasse (One-vs-Rest)
roc_auc = roc_auc_score(
    y_test, y_proba, multi_class="ovr", average="weighted"
)

print(f"Acurácia (treino):    {acc_train:.4f}")
print(f"Acurácia (validação): {acc_val:.4f}")
print(f"Acurácia (teste):     {acc:.4f}")
print(f"Diferença val-teste:  {abs(acc_val - acc):.4f}")
print(f"---")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1:        {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")

Acurácia (treino):    0.9974
Acurácia (validação): 0.9730
Acurácia (teste):     0.9754
Diferença val-teste:  0.0024
---
Precision: 0.9755
Recall:    0.9754
F1:        0.9754
ROC-AUC:   0.9994


### Interpretação: overfitting e consistência

O gap entre treino (0.9974) e validação (0.9730) é de cerca de 2,4 pontos
percentuais — um overfitting leve, esperado para um MLP no MNIST com 49k amostras
de treino. Já a diferença entre validação (0.9730) e teste (0.9754) é de apenas
0.0024, indicando que os dois conjuntos são consistentes entre si.

Para reduzir o gap treino × validação, opções incluem aumentar `alpha` (regularização
L2), usar uma CNN (aproveita a estrutura espacial da imagem) ou aplicar data
augmentation (rotações, deslocamentos). Mantivemos os hiperparâmetros idênticos
ao mini-projeto para reprodutibilidade.

In [65]:
# Relatório detalhado por classe
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      1.00      0.98      1381
           1       0.98      0.99      0.98      1575
           2       0.98      0.97      0.98      1398
           3       0.98      0.97      0.97      1428
           4       0.98      0.97      0.97      1365
           5       0.97      0.97      0.97      1263
           6       0.99      0.98      0.98      1375
           7       0.97      0.97      0.97      1459
           8       0.98      0.95      0.97      1365
           9       0.96      0.98      0.97      1391

    accuracy                           0.98     14000
   macro avg       0.98      0.98      0.98     14000
weighted avg       0.98      0.98      0.98     14000



### Salvando as métricas em `models/metrics.json`

Incluímos:
- Timestamp (UTC) para rastreabilidade
- Versão do scikit-learn — o `.joblib` é sensível a isso
- Hiperparâmetros **lidos do próprio modelo** via `get_params()` — evita divergência
  entre o que está no JSON e o que foi realmente treinado
- `n_iter_` — quantas épocas o early stopping executou

In [66]:
metrics = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "sklearn_version": sklearn.__version__,
    "python_version": platform.python_version(),
    "dataset": "mnist_784",
    "split": {"train": 0.70, "val": 0.10, "test": 0.20, "random_state": 42},
    "model": "MLPClassifier",
    "hyperparameters": {
        "hidden_layer_sizes": list(mlp.hidden_layer_sizes),
        "activation": mlp.activation,
        "solver": mlp.solver,
        "learning_rate_init": mlp.learning_rate_init,
        "alpha": mlp.alpha,
        "max_iter": mlp.max_iter,
        "early_stopping": mlp.early_stopping,
        "random_state": mlp.random_state,
    },
    "n_epochs_executed": int(n_iter),
    "train_metrics": {
        "accuracy": round(acc_train, 4),
    },
    "validation_metrics": {
        "accuracy": round(acc_val, 4),
    },
    "test_metrics": {
        "accuracy": round(acc, 4),
        "precision_weighted": round(precision, 4),
        "recall_weighted": round(recall, 4),
        "f1_weighted": round(f1, 4),
        "roc_auc_ovr_weighted": round(roc_auc, 4),
    },
}

metrics_path = MODELS_DIR / "metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print(f"Métricas salvas em {metrics_path}")
print(json.dumps(metrics, indent=2))

Métricas salvas em models\metrics.json
{
  "timestamp_utc": "2026-10-28T09:29:54.965142+00:00",
  "sklearn_version": "1.9.1",
  "python_version": "3.13.15",
  "dataset": "mnist_784",
  "split": {
    "train": 0.7,
    "val": 0.1,
    "test": 0.2,
    "random_state": 42
  },
  "model": "MLPClassifier",
  "hyperparameters": {
    "hidden_layer_sizes": [
      128
    ],
    "activation": "relu",
    "solver": "adam",
    "learning_rate_init": 0.001,
    "alpha": 0.0001,
    "max_iter": 30,
    "early_stopping": true,
    "random_state": 42
  },
  "n_epochs_executed": 30,
  "train_metrics": {
    "accuracy": 0.9974
  },
  "validation_metrics": {
    "accuracy": 0.973
  },
  "test_metrics": {
    "accuracy": 0.9754,
    "precision_weighted": 0.9755,
    "recall_weighted": 0.9754,
    "f1_weighted": 0.9754,
    "roc_auc_ovr_weighted": 0.9994
  }
}


## 5. Serialização com joblib

O `joblib` é preferível ao `pickle` para objetos do scikit-learn porque:
- Lida melhor com arrays numpy grandes (os pesos da rede neural são arrays)
- É o formato recomendado pela própria documentação do sklearn

**Atenção:** o `.joblib` grava referências ao código (caminhos de import), não o
código em si. Por isso a dependência de `app/preprocessing.py` existir em qualquer
processo que carregue o modelo.

In [67]:
model_path = MODELS_DIR / "mnist_mlp_pipeline.joblib"
joblib.dump(pipeline, model_path)
print(f"Pipeline salvo em {model_path} ({model_path.stat().st_size / 1e6:.2f} MB)")

Pipeline salvo em models\mnist_mlp_pipeline.joblib (2.45 MB)


### Validação da serialização

Para testar de forma honesta que o modelo carrega em **outro processo** (como a API
fará), executamos um subprocess Python limpo via `subprocess`. Diferente de simplesmente
fazer `joblib.load()` neste mesmo kernel (onde `app.preprocessing` já foi importado),
o subprocess começa do zero — se a importação falhar, veremos o erro.

In [68]:
# Teste em processo limpo (subprocess) — simula o que a API fará
resultado = subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "import joblib; "
            f"p = joblib.load('{model_path.as_posix()}'); "
            "print(type(p).__name__)"
        ),
    ],
    capture_output=True,
    text=True,
    cwd=Path.cwd(),  # garante que app/ está no sys.path do subprocess
)

if resultado.returncode == 0:
    print(f"Subprocesso OK — modelo carregado: {resultado.stdout.strip()}")
else:
    print(f"ERRO no subprocesso:\n{resultado.stderr}")

Subprocesso OK — modelo carregado: Pipeline


In [69]:
# Além do subprocesso, validamos a consistência das previsões:
# carregamos do disco e comparamos com o pipeline em memória.
pipeline_carregado = joblib.load(model_path)

# Seleciona 5 amostras de teste (seed fixa para reprodutibilidade)
idx = np.random.RandomState(0).choice(len(X_test), size=5, replace=False)
X_amostras = X_test[idx]
y_amostras_real = y_test[idx]

pred_memoria = pipeline.predict(X_amostras)
pred_carregado = pipeline_carregado.predict(X_amostras)

print("Índice | Real | Em memória | Carregado")
print("-" * 45)
for i, real, p_mem, p_load in zip(idx, y_amostras_real, pred_memoria, pred_carregado):
    flag = "OK" if p_mem == p_load else "DIVERGIU!"
    print(f"{i:>6} | {real:>4} | {p_mem:>10} | {p_load:>9}  [{flag}]")

assert np.array_equal(pred_memoria, pred_carregado), \
    "As previsões divergiram! A serialização falhou."
print("\nSerialização validada: previsões idênticas.")

Índice | Real | Em memória | Carregado
---------------------------------------------
  6038 |    3 |          3 |         3  [OK]
 10758 |    8 |          8 |         8  [OK]
 13882 |    1 |          1 |         1  [OK]
  2760 |    5 |          5 |         5  [OK]
 10778 |    4 |          4 |         4  [OK]

Serialização validada: previsões idênticas.


## 6. Cuidados de reprodutibilidade

1. **Caminhos relativos com `pathlib`** — já feito acima; funciona em qualquer SO.
2. **`random_state=42`** em tudo que é estocástico (split + MLP) — mesma entrada, mesmo resultado.
3. **Versões no `requirements.txt`** — no fim do projeto, após instalar todas as
   dependências da API, rode no terminal (a partir da raiz do repositório):
   ```bash
   pip freeze > requirements.txt
   ```
   Isso congela todas as versões instaladas (inclusive `scikit-learn==`), evitando
   "funciona na minha máquina".
4. **Sensibilidade do joblib à versão do scikit-learn** — um `.joblib` criado com
   sklearn 1.9.0 pode falhar ao carregar em 1.9.1. Por isso salvamos a versão em
   `metrics.json` e fixaremos no `requirements.txt`.

### Pré-requisito

Este notebook deve ser executado com a **raiz do repositório** como diretório de
trabalho, pois o `from app.preprocessing import ...` depende disso.

### Checklist da Fase 1
- [x] Pipeline com normalização interna (`FunctionTransformer` + `app.preprocessing`)
- [x] Divisão estratificada 70/10/20
- [x] MLP treinado com hiperparâmetros do vencedor
- [x] Métricas na validação e no teste (acurácia, P/R/F1 weighted, ROC-AUC OVR)
- [x] `models/metrics.json` salvo (com `n_iter_` e hiperparâmetros do modelo)
- [x] `models/mnist_mlp_pipeline.joblib` salvo e validado (subprocesso limpo + consistência)
- [x] Estrutura de pastas `app/`, `models/`, `data/meus_digitos/` criada

In [70]:
# Resumo final
print("=== Fase 1 concluída ===")
print(f"Modelo:  {model_path}")
print(f"Métricas: {metrics_path}")
print(f"Épocas executadas: {n_iter}")
print(f"Acurácia (treino):    {acc_train:.4f}")
print(f"Acurácia (validação): {acc_val:.4f}")
print(f"Acurácia (teste):     {acc:.4f}")
print(f"F1 weighted:          {f1:.4f}")
print(f"ROC-AUC OVR:          {roc_auc:.4f}")

=== Fase 1 concluída ===
Modelo:  models\mnist_mlp_pipeline.joblib
Métricas: models\metrics.json
Épocas executadas: 30
Acurácia (treino):    0.9974
Acurácia (validação): 0.9730
Acurácia (teste):     0.9754
F1 weighted:          0.9754
ROC-AUC OVR:          0.9994


## 7. Fase 2 — Arquitetura modular

A API não é um arquivo único: cada camada tem uma responsabilidade única.

| Camada | Arquivo | Responsabilidade | Conhece HTTP? |
|---|---|---|---|
| **Schemas** | `app/schemas.py` | Definir a forma do JSON (Pydantic `BaseModel`) | Não |
| **Services** | `app/services.py` | Carregar o `.joblib` e executar a inferência | **Não** |
| **Routers** | `app/routers.py` | Expor endpoints e traduzir erro → status code | Sim |
| **Main** | `app/main.py` | Criar o FastAPI e incluir os routers | Sim |

**Por que o service não conhece HTTP?**
As regras de negócio (carregar modelo, prever) são independentes do protocolo.
Se amanhã você quiser chamar a mesma lógica de um script CLI ou de um notebook,
ela continua funcionando — basta importar `app.services`. Só o `routers.py`
sabe que existe um status 422 ou um `HTTPException`.

**Quem importa quem (sem import circular):**

```
schemas.py   services.py     (independentes, sem importar app.*)
      \         /
       routers.py            (importa os dois)
           |
        main.py              (importa routers)
```

Os dois módulos de domínio são independentes entre si. O `routers.py` importa
os dois, e o `main.py` importa apenas o `routers`. Ninguém importa "para baixo",
logo não há ciclo.

**`app/__init__.py`** transforma `app/` em um pacote Python importável — sem ele,
`import app.main` falharia com `ModuleNotFoundError`.

In [71]:
# Prova que a estrutura existe e que o pacote importa sem erro
import subprocess
import sys
from pathlib import Path

# 1) Árvore de arquivos de app/
print("Árvore de app/:")
for p in sorted(Path("app").rglob("*.py")):
    print(f"  {p}")

# 2) Import limpo em processo novo (simula uvicorn)
resultado = subprocess.run(
    [sys.executable, "-c", "import app.main; print('import app.main OK')"],
    capture_output=True,
    text=True,
    cwd=Path.cwd(),
)
print(resultado.stdout or resultado.stderr)

Árvore de app/:
  app\__init__.py
  app\main.py
  app\preprocessing.py
  app\routers.py
  app\schemas.py
  app\services.py
import app.main OK



## 8. Fase 3 — Ciclo de vida (lifespan)

O modelo `.joblib` (~2.5 MB) deve ser carregado **uma única vez** na subida do
servidor, não a cada requisição. O `lifespan` do FastAPI faz exatamente isso.

### O que acontece antes e depois do `yield`?

```
@asynccontextmanager
async def lifespan(app):
    # ANTES do yield → roda na SUBIDA do servidor
    app.state.modelo = services.carregar_modelo()

    yield          # ← o servidor fica no ar aqui

    # DEPOIS do yield → roda no ENCERRAMENTO (Ctrl+C, SIGTERM)
    app.state.modelo = None
```

### Por que `lifespan` e não `on_event("startup")`?

`on_event` é **deprecated** desde o Starlette 0.26+. O `lifespan` é a API atual,
mais limpa (um único contexto) e com melhor suporte a async.

### Por que `app.state` e não variável global?

`app.state` é um namespace **anexado ao objeto app**. Vantagens:
- Não polui o escopo do módulo
- Pode ser acessado de qualquer lugar via `request.app.state.modelo`
- Fica claro que o estado pertence àquela instância do app

### Trade-off: subir com `modelo = None` vs falhar na inicialização

Optamos por **subir com `None`**:
- O `/health` pode reportar `model_loaded=false` em vez de a API simplesmente não responder.
- O custo é que o erro só aparece em tempo de execução (500 no `/predict`).
- Em produção, um *readiness probe* baseado em `model_loaded` cobriria isso.

A alternativa (falhar cedo) forçaria restart imediato pelo orquestrador —
válida em produção, mas menos útil para desenvolvimento e demonstração.

> **Nota:** usamos `httpx2` (não `httpx`) no `requirements.txt` para evitar o
> aviso de deprecação do Starlette ao usar `TestClient`.

In [72]:
# O `with TestClient(app):` é OBRIGATÓRIO para disparar o lifespan.
# Sem o `with`, o TestClient não executa os blocos antes/depois do yield.
from fastapi.testclient import TestClient
from app.main import app  # o logging já está configurado em main.py

with TestClient(app) as client:
    modelo = app.state.modelo
    print(f"app.state.modelo is not None: {modelo is not None}")
    print(f"tipo: {type(modelo).__name__}")
    print(f"tem predict_proba: {hasattr(modelo, 'predict_proba')}")

print(f"apos o with (servidor parado), modelo: {app.state.modelo}")

INFO: app.services - Modelo carregado (mnist_mlp_pipeline.joblib) em 0.003s (tipo: Pipeline)
INFO: app.main - Modelo liberado.


app.state.modelo is not None: True
tipo: Pipeline
tem predict_proba: True
apos o with (servidor parado), modelo: None


### Prova: `joblib.load` só existe em `services.py`

Nenhum outro módulo chama `joblib.load` diretamente — o carregamento
está encapsulado no service e invocado apenas pelo `lifespan`.

In [73]:
# Busca por joblib.load em todo o pacote app/
from pathlib import Path

for p in sorted(Path("app").rglob("*.py")):
    for i, linha in enumerate(p.read_text(encoding="utf-8").splitlines(), 1):
        if "joblib.load" in linha:
            print(f"{p}:{i}: {linha.strip()}")

app\services.py:48: modelo = joblib.load(MODEL_PATH)


### Demonstração: caminho de falha (modelo ausente)

Apontamos `MODEL_PATH` para um arquivo inexistente com `unittest.mock.patch`
e provamos que o lifespan captura o erro e deixa `app.state.modelo = None` —
a API sobe, mas sem modelo.

> A mensagem `ERROR: app.main - Falha ao carregar modelo` na saída abaixo é
> **esperada** — é exatamente o comportamento documentado no trade-off.

In [74]:
from unittest.mock import patch
from pathlib import Path

from fastapi.testclient import TestClient
from app.main import app
from app import services

caminho_inexistente = Path("models/nao_existe.joblib")

with patch.object(services, "MODEL_PATH", caminho_inexistente):
    with TestClient(app) as client:
        print(f"MODEL_PATH apontado para: {caminho_inexistente}")
        print(f"app.state.modelo is None: {app.state.modelo is None}")
        print("A API subiu, mas sem modelo — /health reportaria model_loaded=false.")

ERROR: app.main - Falha ao carregar modelo: Arquivo de modelo não encontrado: nao_existe.joblib. Execute o notebook da Fase 1 para gerar o .joblib.
INFO: app.main - Modelo liberado.


MODEL_PATH apontado para: models\nao_existe.joblib
app.state.modelo is None: True
A API subiu, mas sem modelo — /health reportaria model_loaded=false.


### Demonstração: arquivo corrompido

Além do arquivo ausente, testamos um `.joblib` com bytes inválidos.
O `try/except` em `services.carregar_modelo` converte o erro em
`ModeloNaoCarregadoError`, e o lifespan captura — a API sobe com `modelo = None`.


> O log da falha é capturado pelo próprio teste e impresso na saída abaixo,
> por isso não aparece como `ERROR` no `stderr` da célula.

In [75]:
import logging
import tempfile
from pathlib import Path
from unittest.mock import patch

from fastapi.testclient import TestClient
from app.main import app
from app import services


class ColetorDeLogs(logging.Handler):
    """Guarda os registros de log em memória, para exibi-los nesta célula."""

    def __init__(self):
        super().__init__(level=logging.INFO)
        self.registros = []

    def emit(self, registro):
        self.registros.append(registro)


# Os handlers do logger raiz escrevem no stderr do Jupyter: se o ERROR passar
# por eles, parece que a célula falhou. Silenciamos só durante o teste e
# guardamos os registros em `coletor`, para imprimir na saída da célula.
raiz = logging.getLogger()
handlers_originais = list(raiz.handlers)
niveis_originais = [h.level for h in handlers_originais]
nivel_raiz_original = raiz.level

coletor = ColetorDeLogs()
raiz.addHandler(coletor)
raiz.setLevel(logging.INFO)
for handler in handlers_originais:
    handler.setLevel(logging.CRITICAL + 1)

try:
    with tempfile.TemporaryDirectory() as d:
        ruim = Path(d) / "corrompido.joblib"
        ruim.write_bytes(b"isto nao e um joblib")
        with patch.object(services, "MODEL_PATH", ruim):
            with TestClient(app):
                assert app.state.modelo is None, "modelo deveria ser None"
finally:
    for handler, nivel in zip(handlers_originais, niveis_originais):
        handler.setLevel(nivel)
    raiz.setLevel(nivel_raiz_original)
    raiz.removeHandler(coletor)

for registro in coletor.registros:
    print(f"{registro.levelname}: {registro.name} - {registro.getMessage()}")

print("Arquivo corrompido: app.state.modelo is None: True")
print("try/except em carregar_modelo funcionou — ModeloNaoCarregadoError capturado.")

ERROR: app.main - Falha ao carregar modelo: Falha ao carregar corrompido.joblib: No module named 'sto nao e um jobli'
INFO: app.main - Modelo liberado.
Arquivo corrompido: app.state.modelo is None: True
try/except em carregar_modelo funcionou — ModeloNaoCarregadoError capturado.


## 9. Fase 4 — Modelagem e validação estrita de dados (Pydantic)

Até aqui `app/schemas.py` tinha só classes vazias (`pass`). Nesta fase ele passa a definir o **contrato de dados** da API: o que ela aceita e o que ela devolve.

### Por que validar antes do modelo?

O MLP espera exatamente 784 números entre 0 e 255. Se alguém enviar 500 pixels, texto ou o valor 300, o modelo ou quebraria (erro 500) ou, pior, devolveria uma previsão sem sentido. Com o Pydantic, o dado inválido é barrado **antes** de chegar ao modelo e o FastAPI responde **422 Unprocessable Entity** automaticamente.

### Decisões de projeto

| Schema | Campo | Restrição | Por quê |
|---|---|---|---|
| `PredictRequest` | `pixels` | lista com `min_length=784` e `max_length=784` | imagem 28x28 achatada; qualquer outro tamanho não serve ao modelo |
| | cada pixel | `int` com `strict=True`, `ge=0`, `le=255` | faixa válida de um pixel; o modo estrito rejeita `"12"`, `12.5`, `12.0` e `True` |
| | (modelo) | `extra="forbid"` | campo desconhecido no JSON vira erro, evitando que um erro de digitação passe despercebido |
| `PredictResponse` | `digito` | `ge=0`, `le=9` | só existem 10 classes |
| | `confianca` | `ge=0`, `le=1` | é uma probabilidade |
| | `probabilidades` | 10 itens, cada um entre 0 e 1 | uma probabilidade por dígito, na ordem 0 a 9 |
| `HealthResponse` | `status`, `model_loaded` | `str` e `bool` | `/health` informa se o modelo está na RAM |
| `ErrorResponse` | `detail` | `str` | formato padrão das respostas de erro |

O arquivo continua sem importar nada de `app.*` nem de HTTP: só Pydantic.

In [76]:
from pathlib import Path

# Código final do módulo de schemas
print(Path("app/schemas.py").read_text(encoding="utf-8"))

"""Modelos Pydantic (schemas) da API.

Responsabilidade: definir a forma EXATA do JSON aceito e retornado.
Pydantic valida automaticamente e gera a documentação OpenAPI (/docs).
Nenhuma regra de negócio ou HTTP deve aparecer aqui.

Contrato de entrada: 784 pixels inteiros (imagem 28x28 achatada), cada um de 0 a 255.
Dados fora desse contrato são barrados pelo Pydantic (o FastAPI responde 422)
ANTES de chegar ao modelo.
"""
from typing import Annotated

from pydantic import BaseModel, ConfigDict, Field

N_PIXELS = 784  # 28 x 28
N_CLASSES = 10  # dígitos 0 a 9

# strict=True: só aceita inteiro de verdade (rejeita "12", 12.5, 12.0 e True).
Pixel = Annotated[int, Field(strict=True, ge=0, le=255)]
Probabilidade = Annotated[float, Field(ge=0.0, le=1.0)]


def _exemplo_pixels() -> list[int]:
    """Imagem de exemplo para o Swagger: um traço vertical (parece um '1')."""
    pixels = [0] * N_PIXELS
    for linha in range(4, 24):
        pixels[linha * 28 + 14] = 255
    return pixels


class P

### Prova 1: o schema de entrada rejeita dados inválidos

Testamos direto o Pydantic (sem precisar de servidor). Para cada caso mostramos se foi aceito e, quando rejeitado, o tipo do erro.

In [77]:
from pydantic import ValidationError
from app.schemas import PredictRequest, PredictResponse, HealthResponse, ErrorResponse

OK = [0] * 784  # imagem válida (tudo preto)


def testar(nome, criar, esperado_ok):
    """Executa `criar()` e compara o resultado com o esperado."""
    try:
        criar()
        aceito, motivo = True, ""
    except ValidationError as exc:
        aceito, motivo = False, exc.errors()[0]["type"]
    status = "ACEITO   " if aceito else "REJEITADO"
    veredito = "OK" if aceito == esperado_ok else "FALHOU!"
    print(f"{status} | {nome:<34} {motivo:<22} [{veredito}]")
    assert aceito == esperado_ok, f"Resultado inesperado em: {nome}"


casos_entrada = [
    ("784 pixels válidos",              lambda: PredictRequest(pixels=OK), True),
    ("783 pixels (faltando 1)",         lambda: PredictRequest(pixels=[0] * 783), False),
    ("785 pixels (sobrando 1)",         lambda: PredictRequest(pixels=[0] * 785), False),
    ("pixel -1 (abaixo do mínimo)",     lambda: PredictRequest(pixels=[-1] + [0] * 783), False),
    ("pixel 256 (acima do máximo)",     lambda: PredictRequest(pixels=[256] + [0] * 783), False),
    ("pixel texto 'abc'",               lambda: PredictRequest(pixels=["abc"] + [0] * 783), False),
    ("pixel texto numérico '12'",       lambda: PredictRequest(pixels=["12"] + [0] * 783), False),
    ("pixel float 12.5",                lambda: PredictRequest(pixels=[12.5] + [0] * 783), False),
    ("pixel booleano True",             lambda: PredictRequest(pixels=[True] + [0] * 783), False),
    ("pixel nulo (None)",               lambda: PredictRequest(pixels=[None] + [0] * 783), False),
    ("pixels não é lista (texto)",      lambda: PredictRequest(pixels="abc"), False),
    ("campo extra no JSON",             lambda: PredictRequest(pixels=OK, extra=1), False),
    ("campo pixels ausente",            lambda: PredictRequest(), False),
]

for nome, criar, esperado in casos_entrada:
    testar(nome, criar, esperado)

print("\nTodos os casos de entrada se comportaram como esperado.")

ACEITO    | 784 pixels válidos                                        [OK]
REJEITADO | 783 pixels (faltando 1)            too_short              [OK]
REJEITADO | 785 pixels (sobrando 1)            too_long               [OK]
REJEITADO | pixel -1 (abaixo do mínimo)        greater_than_equal     [OK]
REJEITADO | pixel 256 (acima do máximo)        less_than_equal        [OK]
REJEITADO | pixel texto 'abc'                  int_type               [OK]
REJEITADO | pixel texto numérico '12'          int_type               [OK]
REJEITADO | pixel float 12.5                   int_type               [OK]
REJEITADO | pixel booleano True                int_type               [OK]
REJEITADO | pixel nulo (None)                  int_type               [OK]
REJEITADO | pixels não é lista (texto)         list_type              [OK]
REJEITADO | campo extra no JSON                extra_forbidden        [OK]
REJEITADO | campo pixels ausente               missing                [OK]

Todos os casos de entrad

### Prova 2: os schemas de saída também têm restrições

A resposta também é um contrato: se o código da Fase 5 tentar devolver algo fora dele (por exemplo, dígito 10), o erro aparece no desenvolvimento em vez de chegar ao cliente.

In [78]:
probs = [0.1] * 10

casos_saida = [
    ("PredictResponse válida",          lambda: PredictResponse(digito=3, confianca=0.1, probabilidades=probs), True),
    ("digito = 10 (fora de 0-9)",       lambda: PredictResponse(digito=10, confianca=0.1, probabilidades=probs), False),
    ("digito = -1",                     lambda: PredictResponse(digito=-1, confianca=0.1, probabilidades=probs), False),
    ("confianca = 1.5 (> 1)",           lambda: PredictResponse(digito=3, confianca=1.5, probabilidades=probs), False),
    ("probabilidades com 9 itens",      lambda: PredictResponse(digito=3, confianca=0.1, probabilidades=[0.1] * 9), False),
    ("uma probabilidade = 1.2",         lambda: PredictResponse(digito=3, confianca=0.1, probabilidades=[1.2] + [0.0] * 9), False),
    ("HealthResponse válida",           lambda: HealthResponse(status="ok", model_loaded=True), True),
    ("HealthResponse sem model_loaded", lambda: HealthResponse(status="ok"), False),
    ("ErrorResponse válida",            lambda: ErrorResponse(detail="Algo deu errado."), True),
]

for nome, criar, esperado in casos_saida:
    testar(nome, criar, esperado)

print("\nTodos os casos de saída se comportaram como esperado.")

ACEITO    | PredictResponse válida                                    [OK]
REJEITADO | digito = 10 (fora de 0-9)          less_than_equal        [OK]
REJEITADO | digito = -1                        greater_than_equal     [OK]
REJEITADO | confianca = 1.5 (> 1)              less_than_equal        [OK]
REJEITADO | probabilidades com 9 itens         too_short              [OK]
REJEITADO | uma probabilidade = 1.2            less_than_equal        [OK]
ACEITO    | HealthResponse válida                                     [OK]
REJEITADO | HealthResponse sem model_loaded    missing                [OK]
ACEITO    | ErrorResponse válida                                      [OK]

Todos os casos de saída se comportaram como esperado.


### Prova 3: o FastAPI transforma a falha de validação em HTTP 422

As rotas reais só chegam na Fase 5. Para provar desde já o comportamento, montamos abaixo um **mini-app descartável** (só neste notebook, não faz parte do projeto) que usa os schemas. O `TestClient` simula as requisições.

Repare que a função da rota **nunca é executada** quando o dado é inválido: o 422 sai antes.

In [79]:
import logging

from fastapi import FastAPI
from fastapi.testclient import TestClient

# Deixa a saída desta célula limpa: sem as linhas INFO do httpx2.
logging.getLogger("httpx2").setLevel(logging.WARNING)

mini_app = FastAPI()
chamadas = {"rota": 0}


@mini_app.post("/teste", response_model=PredictResponse)
def rota_de_teste(corpo: PredictRequest):
    chamadas["rota"] += 1  # conta quantas vezes o corpo da rota realmente rodou
    return PredictResponse(digito=1, confianca=0.5, probabilidades=[0.1] * 10)


client_teste = TestClient(mini_app)

requisicoes = [
    ("válida",                  {"pixels": OK}),
    ("poucos pixels",           {"pixels": [0] * 5}),
    ("pixel 300",               {"pixels": [300] + [0] * 783}),
    ("pixel texto",             {"pixels": ["a"] + [0] * 783}),
    ("campo extra",             {"pixels": OK, "extra": 1}),
    ("corpo sem pixels",        {}),
]

for nome, corpo in requisicoes:
    resp = client_teste.post("/teste", json=corpo)
    print(f"{nome:<18} -> HTTP {resp.status_code}")

print(f"\nA rota executou {chamadas['rota']} vez(es): só a requisição válida chegou até ela.")

válida             -> HTTP 200
poucos pixels      -> HTTP 422
pixel 300          -> HTTP 422
pixel texto        -> HTTP 422
campo extra        -> HTTP 422
corpo sem pixels   -> HTTP 422

A rota executou 1 vez(es): só a requisição válida chegou até ela.


### Resumo da Fase 4

- `PredictRequest`: 784 inteiros de 0 a 255, modo estrito e `extra="forbid"`.
- `PredictResponse`: dígito (0-9), confiança (0-1) e 10 probabilidades.
- `HealthResponse` e `ErrorResponse`: contratos de saúde e de erro.
- Dados inválidos geram **422** automaticamente, antes de tocar no modelo.
- O Swagger (`/docs`) mostrará descrição, limites e um exemplo (um traço vertical, parecido com o dígito 1) para cada campo

